In [292]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.experiment_runner.models import ExperimentRun, RequestRecord

In [293]:
GROUP_DIR = Path("../assets/experiments/error_new6")

run_dirs = [p for p in GROUP_DIR.iterdir() if p.is_dir()]
print("Runs:", len(run_dirs))

Runs: 7


In [294]:
def load_run(path: Path) -> ExperimentRun:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    reqs = [RequestRecord(**r) for r in data["requests"]]
    data["requests"] = reqs

    return ExperimentRun(**data)

In [295]:
def compute_wa(run: ExperimentRun) -> float:
    replicas = [
        len(r.upstream.get("sockets", [])) or 1
        for r in run.requests
    ]
    return float(np.mean(replicas))

In [296]:
def extract_row(run: ExperimentRun, group: str) -> dict:
    balancer = run.factors.get("balancer")
    replication = run.factors.get("replication") or "none"
    seed = run.factors.get("seed", -1)

    latency = run.summary["overall"]["latency_ms"]

    p99 = latency["p99"]
    p95 = latency["p95"]
    wa = compute_wa(run)

    return {
        "run_id": run.run_id,
        "seed": seed,
        "algorithm": balancer,
        "strategy": replication,
        "group": group,
        "p99": p99,
        "p95": p95,
        "wa": wa,
    }

In [297]:
def collect_rows(root: Path) -> pd.DataFrame:
    rows = []

    for file in root.rglob("*.json"):
        if "baseline" in file.parts:
            group = "baseline"
        elif "non_adaptive" in file.parts:
            group = "non_adaptive"
        elif "adaptive" in file.parts:
            group = "adaptive"
        else:
            continue

        run = load_run(file)
        row = extract_row(run, group)

        # batch_id = timestamp папки эксперимента
        row["batch_id"] = file.parents[1].name
        row["file_name"] = file.name

        rows.append(row)

    return pd.DataFrame(rows)

In [298]:
def aggregate(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df
        .groupby(["algorithm", "strategy", "group"], as_index=False)
        .agg({
            "p99": "mean",
            "wa": "mean",
        })
    )

In [299]:
def compute_scores(df: pd.DataFrame) -> pd.DataFrame:
    results = []

    for group in ["non_adaptive", "adaptive"]:
        repl = df[df.group == group].copy()
        base = df[df.group == "baseline"].copy()

        if repl.empty:
            continue

        # Парное сравнение по algorithm + seed
        merged = repl.merge(
            base,
            on=["algorithm", "seed"],
            suffixes=("", "_baseline"),
        )

        if merged.empty:
            continue

        merged["p99_improvement_percent"] = (
                (merged["p99_baseline"] - merged["p99"]) / merged["p99_baseline"] * 100
        )

        merged["p95_improvement_percent"] = (
                (merged["p95_baseline"] - merged["p95"]) / merged["p95_baseline"] * 100
        )

        merged["wa_ratio"] = merged["wa"] / merged["wa_baseline"]

        merged["efficiency"] = (
                merged["p99_improvement_percent"] / merged["wa_ratio"]
        )

        merged["group"] = group

        agg = (
            merged
            .groupby(["algorithm", "strategy", "group"], as_index=False)
            .agg(
                p99_improvement_percent=("p99_improvement_percent", "mean"),
                p99_median=("p99_improvement_percent", "median"),
                p99_min=("p99_improvement_percent", "min"),
                p99_max=("p99_improvement_percent", "max"),
                p99_std=("p99_improvement_percent", "std"),

                p95_improvement_percent=("p95_improvement_percent", "mean"),
                p95_median=("p95_improvement_percent", "median"),
                p95_min=("p95_improvement_percent", "min"),
                p95_max=("p95_improvement_percent", "max"),
                p95_std=("p95_improvement_percent", "std"),

                wa=("wa_ratio", "mean"),
                efficiency=("efficiency", "mean"),
                experiments_count=("seed", "count"),
                positive_p99_rate=("p99_improvement_percent", lambda s: float((s > 0).mean())),
                positive_p95_rate=("p95_improvement_percent", lambda s: float((s > 0).mean())),
            )
        )

        ci_half_p99 = 1.96 * agg["p99_std"].fillna(0.0) / np.sqrt(agg["experiments_count"])
        ci_half_p95 = 1.96 * agg["p95_std"].fillna(0.0) / np.sqrt(agg["experiments_count"])

        low_p99 = agg["p99_improvement_percent"] - ci_half_p99
        high_p99 = agg["p99_improvement_percent"] + ci_half_p99
        agg["p99_ci95_range"] = (
                low_p99.round(2).astype(str) + " – " + high_p99.round(2).astype(str)
        )

        low_p95 = agg["p95_improvement_percent"] - ci_half_p95
        high_p95 = agg["p95_improvement_percent"] + ci_half_p95
        agg["p95_ci95_range"] = (
                low_p95.round(2).astype(str) + " – " + high_p95.round(2).astype(str)
        )

        agg["score"] = (
                agg["p99_improvement_percent"] / (agg["wa"] * (1 + ci_half_p99))
        )

        results.append(agg)

    if not results:
        return pd.DataFrame()

    final = pd.concat(results, ignore_index=True)
    final = final.sort_values(
        by=["algorithm", "group", "p99_improvement_percent"],
        ascending=[True, True, False],
    )
    return final

In [300]:
def build_paired_comparison(df: pd.DataFrame, group: str = "non_adaptive") -> pd.DataFrame:
    repl = df[df.group == group].copy()
    base = df[df.group == "baseline"].copy()

    merged = repl.merge(
        base,
        on=["algorithm", "seed"],
        suffixes=("", "_baseline"),
    )

    if merged.empty:
        return pd.DataFrame()

    merged["delta_p99"] = merged["p99_baseline"] - merged["p99"]
    merged["delta_p95"] = merged["p95_baseline"] - merged["p95"]

    merged["p99_improvement_percent"] = (
            merged["delta_p99"] / merged["p99_baseline"] * 100
    )
    merged["p95_improvement_percent"] = (
            merged["delta_p95"] / merged["p95_baseline"] * 100
    )

    cols = [
        "algorithm",
        "strategy",
        "seed",
        "p99_baseline",
        "p99",
        "delta_p99",
        "p99_improvement_percent",
        "p95_baseline",
        "p95",
        "delta_p95",
        "p95_improvement_percent",
        "wa_baseline",
        "wa",
    ]
    return merged[cols].sort_values(["algorithm", "strategy", "seed"])

In [301]:
def analyze_all(root: Path):
    df = collect_rows(root)
    paired = build_paired_comparison(df, group="non_adaptive")

    if df.empty:
        raise ValueError("Нет данных")

    return compute_scores(df), paired

In [302]:
res, paired = analyze_all(GROUP_DIR)

In [303]:
print(res)
res

  algorithm strategy         group  p99_improvement_percent  p99_median  \
0    topsis   hedged  non_adaptive                12.659122   12.470526   

     p99_min   p99_max   p99_std  p95_improvement_percent  p95_median  ...  \
0  11.257843  14.60841  1.435793                 0.288294   -0.095788  ...   

    p95_max   p95_std        wa  efficiency  experiments_count  \
0  9.487883  5.388599  1.491167    8.488439                  6   

   positive_p99_rate  positive_p95_rate  p99_ci95_range p95_ci95_range  \
0                1.0                0.5   11.51 – 13.81    -4.02 – 4.6   

      score  
0  3.950632  

[1 rows x 21 columns]


,algorithm,strategy,group,p99_improvement_percent,p99_median,p99_min,p99_max,p99_std,p95_improvement_percent,p95_median,...,p95_max,p95_std,wa,efficiency,experiments_count,positive_p99_rate,positive_p95_rate,p99_ci95_range,p95_ci95_range,score
0,topsis,hedged,non_adaptive,12.659122,12.470526,11.257843,14.60841,1.435793,0.288294,-0.095788,...,9.487883,5.388599,1.491167,8.488439,6,1.0,0.5,11.51 – 13.81,-4.02 – 4.6,3.950632


In [304]:
print(paired)
paired

  algorithm strategy   seed  p99_baseline           p99    delta_p99  \
0    topsis   hedged  10000  18343.505233  15914.323319  2429.181914   
1    topsis   hedged  10001  18112.822005  15466.826738  2645.995267   
2    topsis   hedged  10002  18197.317850  16148.692370  2048.625480   
3    topsis   hedged  10003  18573.551798  16400.758818  2172.792980   
4    topsis   hedged  10004  18279.020410  15747.088650  2531.931760   
5    topsis   hedged  10005  18177.379970  16124.090095  2053.289875   

   p99_improvement_percent  p95_baseline           p95    delta_p95  \
0                13.242736  10347.228790  10243.137155   104.091635   
1                14.608410  10373.067555  10250.703185   122.364370   
2                11.257843   9968.882500  10088.266080  -119.383580   
3                11.698317  10112.667745  10818.398730  -705.730985   
4                13.851572  11243.115880  10176.382160  1066.733720   
5                11.295852  10684.550155  10873.399055  -188.848900  

,algorithm,strategy,seed,p99_baseline,p99,delta_p99,p99_improvement_percent,p95_baseline,p95,delta_p95,p95_improvement_percent,wa_baseline,wa
0,topsis,hedged,10000,18343.505233,15914.323319,2429.181914,13.242736,10347.228790,10243.137155,104.091635,1.005986,1.0,1.500667
1,topsis,hedged,10001,18112.822005,15466.826738,2645.995267,14.608410,10373.067555,10250.703185,122.364370,1.179635,1.0,1.488000
2,topsis,hedged,10002,18197.317850,16148.692370,2048.625480,11.257843,9968.882500,10088.266080,-119.383580,-1.197562,1.0,1.480000
3,topsis,hedged,10003,18573.551798,16400.758818,2172.792980,11.698317,10112.667745,10818.398730,-705.730985,-6.978683,1.0,1.499000
4,topsis,hedged,10004,18279.020410,15747.088650,2531.931760,13.851572,11243.115880,10176.382160,1066.733720,9.487883,1.0,1.492667
5,topsis,hedged,10005,18177.379970,16124.090095,2053.289875,11.295852,10684.550155,10873.399055,-188.848900,-1.767495,1.0,1.486667


In [305]:
def collect_request_level(root: Path) -> pd.DataFrame:
    rows = []

    for file in root.rglob("*.json"):
        # определяем группу
        if "baseline" in file.parts:
            group = "baseline"
        elif "non_adaptive" in file.parts:
            group = "non_adaptive"
        elif "adaptive" in file.parts:
            group = "adaptive"
        else:
            continue

        run = load_run(file)

        algorithm = run.factors.get("balancer")
        strategy = run.factors.get("replication") or "none"

        for r in run.requests:
            rows.append({
                "algorithm": algorithm,
                "strategy": strategy,
                "group": group,
                "ok": r.ok,
            })

    return pd.DataFrame(rows)

In [306]:
df_req = collect_request_level(GROUP_DIR)

error_stats = (
    df_req
    .groupby(["algorithm", "strategy", "group"])["ok"]
    .agg(
        total="count",
        success="sum"
    )
)

error_stats["errors"] = error_stats["total"] - error_stats["success"]

error_stats["error_rate"] = (
        error_stats["errors"] / error_stats["total"]
)

error_stats["error_rate_percent"] = (
        error_stats["error_rate"] * 100
).round(2)

error_stats.sort_values("error_rate", ascending=False)
print(error_stats)

                                 total  success  errors  error_rate  \
algorithm strategy group                                              
topsis    hedged   non_adaptive  18000    17023     977    0.054278   
          none     baseline      18000    16484    1516    0.084222   

                                 error_rate_percent  
algorithm strategy group                             
topsis    hedged   non_adaptive                5.43  
          none     baseline                    8.42  
